## Module 1: Introduction to Database Interaction with Python

Welcome to NextNexa! In this module, we'll explore the fundamental concepts of database interaction using Python, focusing on SQLite for local data management. We'll cover database creation, data insertion, retrieval, and a progressive approach to building intelligent agents that can interpret natural language queries for database operations.

### 1.1 Setting up Our Database: Creation and Initial Data Population

Before we can query data, we need a database. This section demonstrates how to create an SQLite database named `demo.db` and initialize a table named `sales`. We will then populate this table with initial product sales data. SQLite is an excellent choice for embedded and lightweight database applications, making it ideal for prototyping and local development.

In [ ]:
import sqlite3

# Connect to the SQLite database. If 'demo.db' does not exist, it will be created.
conn = sqlite3.connect("demo.db")
cursor = conn.cursor()

# Create the 'sales' table if it doesn't already exist.
# 'id' is an auto-incrementing primary key for unique identification.
# 'product' stores the name of the product as text.
# 'quantity' stores the number of units sold as an integer.
# 'price' stores the price per unit as a real number (floating-point).
cursor.execute("""
CREATE TABLE IF NOT EXISTS sales (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    product TEXT,
    quantity INTEGER,
    price REAL
)
""")

# Insert multiple rows of data into the 'sales' table.
# The '?' are placeholders for the values provided in the list of tuples.
cursor.executemany("""
INSERT INTO sales (product, quantity, price)
VALUES (?, ?, ?)
""", [
    ("Laptop", 2, 1200),
    ("Mouse", 10, 25),
    ("Keyboard", 5, 45),
    ("Monitor", 3, 300)
])

# Commit the changes to the database to make them permanent.
conn.commit()
# Close the database connection.
conn.close()

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("demo.db")

df = pd.read_sql_query("SELECT * FROM sales", conn)
df


### 1.2 Inspecting Our Database: Data Retrieval with Pandas

With our database populated, the next logical step is to verify the data's integrity and structure. Here, we utilize the `pandas` library, a cornerstone for data manipulation in Python, to connect to our SQLite database and retrieve all records from the `sales` table. This provides a clear tabular view of our data, confirming the successful insertion and schema.

In [ ]:
import sqlite3
import pandas as pd

# Establish a connection to the SQLite database 'demo.db'.
conn = sqlite3.connect("demo.db")

# Use pandas to read data directly from the SQL query.
# pd.read_sql_query executes the SQL query and returns the results as a DataFrame.
# The query selects all columns and rows from the 'sales' table.
df = pd.read_sql_query("SELECT * FROM sales", conn)

# Display the entire DataFrame 'df'.
# In Colab, simply putting the DataFrame at the end of a cell will display it nicely.
df

# Close the database connection once the data has been retrieved.
conn.close()

### 1.3 Introducing the Micro-Agent Communication Protocol (MCP) Server

To enable sophisticated natural language interaction with our database, we'll implement a Micro-Agent Communication Protocol (MCP) server. This server will host a set of tools (functions) that our AI agent can call to perform database operations. This design pattern separates the AI logic from the data access layer, promoting modularity and scalability. The `%%writefile` magic command is used to create a Python script named `rdbms_mcp_server.py` containing our server logic. The `FastMCP` framework simplifies the creation of these tool-based servers.

*   **`run_sql(query: str)`**: A utility function to execute raw SQL queries against our `demo.db` and return structured results including columns and rows.
*   **`get_tables() -> List[str]`**: A tool that queries the `sqlite_master` table to list all available tables in the database.
*   **`run_query(query: str) -> Dict[str, Any]`**: A general-purpose tool to execute any provided SQL query and return its results.
*   **`get_sales_summary() -> Dict[str, Any]`**: A specific tool designed to calculate and return the total revenue and total items sold from the `sales` table.
*   The `if __name__ == "__main__"` block ensures that the server can be run directly using `uvicorn`.

In [ ]:
# Execute the rdbms_mcp_server.py script using uvicorn.
# `nohup` ensures the process continues to run even if the shell exits.
# `&` runs the command in the background, preventing it from blocking the notebook.
# The server will listen on host 0.0.0.0 (all available network interfaces) and port 8000.
!nohup uvicorn rdbms_mcp_server:app --host 0.0.0.0 --port 8000 &

### 1.5 Testing the MCP Server Tools from the Client

Now that our MCP server is running, we can import and directly invoke the tools it exposes. This demonstrates the client-side interaction with our defined database operations, bypassing the natural language interface for direct function calls. This is essential for verifying the server's functionality before integrating it with an AI model.

*   **`from rdbms_mcp_server import get_tables, run_query, get_sales_summary`**: Imports the specific tool functions directly from our server script (which is now available as a Python module).
*   **`print("Tables:", get_tables())`**: Calls the `get_tables` tool and prints the list of tables found in `demo.db`.
*   **`print("\nSales Data:")`**: Prints a header for the sales data.
*   **`print(run_query("SELECT * FROM sales"))`**: Executes a raw SQL query via the `run_query` tool to retrieve all sales records.
*   **`print("\nSales Summary:")`**: Prints a header for the sales summary.
*   **`print(get_sales_summary())`**: Calls the `get_sales_summary` tool to retrieve aggregated sales information.

In [ ]:
%%writefile rdbms_mcp_server.py
import sqlite3
from typing import List, Dict, Any
from mcp.server.fastmcp import FastMCP

# Initialize the FastMCP application with a service name.
app = FastMCP("rdbms-assistant")

# Define the path to our SQLite database.
DB_PATH = "demo.db"

# Helper function to execute raw SQL queries and return structured results.
def run_sql(query: str):
    # Establish a connection to the database.
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    # Execute the provided SQL query.
    cursor.execute(query)
    # Fetch all results (rows) from the executed query.
    rows = cursor.fetchall()
    # Extract column names from the cursor description. This provides metadata about the query results.
    columns = [desc[0] for desc in cursor.description] if cursor.description else []
    # Close the database connection.
    conn.close()
    # Return results as a dictionary containing column names and rows.
    return {"columns": columns, "rows": rows}

# MCP tool to retrieve a list of all table names in the database.
@app.tool()
def get_tables() -> List[str]:
    """Return list of tables in the database."""
    # Query sqlite_master table for names of tables.
    result = run_sql("SELECT name FROM sqlite_master WHERE type='table'")
    # Extract table names from the query result.
    return [row[0] for row in result["rows"]]

# MCP tool to run any given SQL query and return its results.
@app.tool()
def run_query(query: str) -> Dict[str, Any]:
    """Run a SQL query and return results."""
    # Simply delegate to the run_sql helper function.
    return run_sql(query)

# MCP tool to calculate and return the total revenue and total items sold from the 'sales' table.
@app.tool()
def get_sales_summary() -> Dict[str, Any]:
    """Return total revenue and total items sold."""
    # Execute a SQL query to sum up (quantity * price) for total revenue and quantity for total items.
    result = run_sql("""
        SELECT
            SUM(quantity * price) AS total_revenue,
            SUM(quantity) AS total_items
        FROM sales
    """)
    # Extract the first (and only) row of the aggregated results.
    row = result["rows"][0]
    # Return a dictionary with labeled total revenue and total items.
    return {
        "total_revenue": row[0],
        "total_items": row[1]
    }

# Standard Python entry point: if this script is run directly, start the uvicorn server.
if __name__ == "__main__":
    import uvicorn
    # Run the FastMCP application using uvicorn, listening on all interfaces (0.0.0.0) on port 8000.
    uvicorn.run(app, host="0.0.0.0", port=8000)

### 1.4 Launching the MCP Server in the Background

This cell initiates our `rdbms_mcp_server.py` using `uvicorn` to run it as a web server. The `!nohup ... &` command allows the server to run in the background, detached from the current terminal session, which is crucial for continuous operation without blocking the notebook. The server listens on `0.0.0.0:8000`.

*   **`!nohup uvicorn rdbms_mcp_server:app --host 0.0.0.0 --port 8000 &`**: Executes the server script using `uvicorn`. `nohup` prevents the process from stopping if the shell is closed, and `&` runs it in the background.
*   **Note on `Address already in use`**: If the server was previously started and not properly shut down, you might encounter an "address already in use" error. This simply means the port is occupied, and the server is likely still running from a previous execution. In a production environment, proper process management (e.g., stopping existing processes) would be required.

In [ ]:
# This cell is an alternative way to run the server, but it blocks the current cell execution.
# It's primarily used for debugging or when you want the server to run in the foreground.
# In this scenario, it is likely to produce an 'address already in use' error if the `nohup` command was successful,
# as the server would already be running on port 8000.
!python rdbms_mcp_server.py

In [ ]:
from rdbms_mcp_server import get_tables, run_query, get_sales_summary

# Call the 'get_tables' tool to retrieve a list of tables in the database and print the result.
print("Tables:", get_tables())

# Print a descriptive header for the sales data.
print("\nSales Data:")
# Call the 'run_query' tool with a SQL SELECT statement to fetch all records from the 'sales' table.
# The result, which includes columns and rows, is then printed.
print(run_query("SELECT * FROM sales"))

# Print a descriptive header for the sales summary.
print("\nSales Summary:")
# Call the 'get_sales_summary' tool to retrieve aggregated sales metrics (total revenue, total items).
# The returned dictionary containing these metrics is then printed.
print(get_sales_summary())

## Module 2: Building a Basic Natural Language Interface for Database Queries

In this module, we advance our database interaction by developing a natural language (NL) interface. We start with a foundational `ask_database` function that can interpret simple textual commands and map them to our existing database tools. This lays the groundwork for more sophisticated AI-driven query processing.

### 2.1 Importing Necessary Components

Before we build our natural language processing (NLP) layer, we need to import the essential components. This includes the database interaction tools from our `rdbms_mcp_server` and the `ai` module from `google.colab`, which provides access to powerful generative AI capabilities for understanding and generating text.

In [ ]:
# Import the database interaction tools directly from our rdbms_mcp_server script.
# These functions (get_tables, run_query, get_sales_summary) are defined as MCP tools but can be called locally.
from rdbms_mcp_server import get_tables, run_query, get_sales_summary

# Import the `ai` module from `google.colab` to access generative AI capabilities.
# This module will be used for natural language processing and text generation.
from google.colab import ai

### 2.2 Initial `ask_database` Function: Simple Keyword-Based Tool Dispatch

This `ask_database` function represents our first attempt at an NL interface. It uses simple keyword matching to determine which underlying database tool (`get_tables`, `get_sales_summary`, `run_query`) to invoke. If no specific keyword is matched, it defers to a general AI model (`ai.generate_text`) for a more open-ended response.

*   **`question_lower = question.lower()`**: Converts the input question to lowercase for case-insensitive keyword matching.
*   **`if "table" in question_lower:`**: Checks if the question contains "table" to call `get_tables()`.
*   **`if "total revenue" in question_lower or "sales summary" in question_lower:`**: Checks for revenue-related keywords to call `get_sales_summary()`.
*   **`if "select" in question_lower:`**: If "select" is present, it directly passes the entire question string to `run_query()`, assuming the user has provided a valid SQL `SELECT` statement.
*   **`return ai.generate_text(question)`**: As a fallback, if no specific tool is matched, the original question is sent to the AI model for a general explanation or answer.

In [ ]:
from google.colab import ai


### 2.3 Testing the Initial `ask_database` Function

Here, we test the capabilities of our initial `ask_database` function with a variety of natural language queries. This helps us understand its current strengths (simple tool dispatch) and limitations (lack of robust SQL generation).

*   **`print(ask_database("What tables are in the database"))`**: Demonstrates calling `get_tables`.
*   **`print(ask_database("What is the total revenue"))`**: Demonstrates calling `get_sales_summary`.
*   **`print(ask_database("SELECT * FROM sales WHERE price > 100"))`**: This query includes "SELECT", so it directly calls `run_query`. It works because the input is already valid SQL.
*   **`print(ask_database("Explain what a database is"))`**: This query doesn't match any specific tool, so it falls back to `ai.generate_text` for a general explanation.

In [ ]:
def ask_database(question):
    # Convert the incoming question to lowercase for case-insensitive matching.
    question_lower = question.lower()

    # Check for keywords to dispatch to specific database tools.
    # If 'table' is in the question, call the `get_tables` tool.
    if "table" in question_lower:
        return get_tables()

    # If 'total revenue' or 'sales summary' is in the question, call the `get_sales_summary` tool.
    if "total revenue" in question_lower or "sales summary" in question_lower:
        return get_sales_summary()

    # If 'select' is in the question, assume the user is providing a raw SQL query.
    # Pass the entire question directly to the `run_query` tool for execution.
    if "select" in question_lower:
        return run_query(question)

    # If no specific tool is matched by keywords, fall back to the generative AI model.
    # The AI model will attempt to generate a general text response to the question.
    return ai.generate_text(question)

### 2.4 Identifying Limitations: SQL Error with Complex Natural Language

This cell highlights a critical limitation of our simple keyword-based `ask_database` function. When presented with a complex natural language question that includes the word "select" but is not a valid SQL query, the function attempts to execute the entire natural language string as SQL, leading to an `OperationalError`.

*   **`print(ask_database("what is price of laptop select from database and give me result"))`**: This input contains "select", triggering `run_query(question)`. However, `"what is price of laptop select from database and give me result"` is not valid SQL, causing the database to raise an `OperationalError`.

In [ ]:
# Test 1: Query for tables in the database.
print(ask_database("What tables are in the database"))

# Test 2: Query for the total revenue (sales summary).
print(ask_database("What is the total revenue"))

# Test 3: Provide a valid SQL query directly.
# This should execute via the `run_query` tool because of the 'SELECT' keyword.
print(ask_database("SELECT * FROM sales WHERE price > 100"))

# Test 4: Ask a general question that doesn't match any specific tool.
# This should fall back to the `ai.generate_text` for a general explanation.
print(ask_database("Explain what a database is"))

### 2.5 Enhancing NL-to-SQL Translation: `nl_to_sql` and `clean_sql`

To overcome the SQL error issue, we need a robust mechanism to translate natural language into accurate SQL queries. This section introduces two helper functions: `nl_to_sql`, which uses an AI model for translation, and `clean_sql`, which post-processes the AI-generated SQL to ensure it's syntactically correct and free of extraneous characters.

In [ ]:
# This query demonstrates a limitation: 'select' is in the question, but the full string is not valid SQL.
# Our current `ask_database` will attempt to execute the entire string as an SQL query,
# resulting in an `OperationalError` because it's not syntactically correct SQL.
print(ask_database("what is price of laptop select from database and give me result"))

Generated SQL: ```sql
SELECT price FROM sales WHERE product = 'laptop';
```
Cleaned SQL: SELECT price FROM sales WHERE product = 'laptop';


NameError: name 'answer_with_data' is not defined

In [ ]:
def clean_sql(sql):
    # Remove leading/trailing whitespace from the generated SQL string.
    sql = sql.strip()
    # Remove common markdown fence for SQL blocks (e.g., '```sql').
    sql = sql.replace("```sql", "")
    # Remove generic markdown fence (e.g., '```').
    sql = sql.replace("```", "")
    # Return the cleaned SQL string, ensuring any new leading/trailing whitespace from replacements is removed.
    return sql.strip()

### 2.5.1 AI-Powered Natural Language to SQL Translation: The `nl_to_sql` Function

This `nl_to_sql` function is the core of our enhanced natural language understanding. It leverages a generative AI model (`ai.generate_text`) to convert arbitrary natural language questions into precise SQLite SQL queries. The prompt is carefully crafted to guide the AI in generating only SQL, based on the provided table schema.

In [ ]:
# The `ai` module from `google.colab` is already imported in cell S0EjKUaM50Ln.
# from google.colab import ai

def nl_to_sql(question):
    # Define a detailed prompt for the generative AI model.
    # The prompt instructs the AI to convert a natural language question into a SQLite SQL query.
    # It explicitly provides the table schema ('sales' table with 'id', 'product', 'quantity', 'price' columns).
    # Crucially, it instructs the AI to *only* return the SQL query and no additional explanations.
    prompt = f"""
    Convert this question into a valid SQLite SQL query.
    Table: sales(id, product, quantity, price)
    Only return SQL. No explanation.

    Question: {question}
    """
    # Call the AI model to generate text based on the prompt.
    # `.strip()` is used to remove any leading/trailing whitespace that the AI model might include.
    return ai.generate_text(prompt).strip()

### 2.6 Refined `ask_database` Function with NL-to-SQL and Error Handling

This iteration of the `ask_database` function integrates our new `nl_to_sql` and `clean_sql` functions. It now intelligently translates natural language questions into SQL, attempts to execute them, and provides basic error handling for SQL execution failures. This significantly improves its ability to respond to diverse queries.

*   **`sql = nl_to_sql(question)`**: The first step is to convert the natural language question into a SQL query using our AI-powered `nl_to_sql` function.
*   **`print("Generated SQL:", sql)`**: Logs the raw SQL generated by the AI for debugging and transparency.
*   **`sql = clean_sql(sql)`**: Cleans the AI-generated SQL.
*   **`print("Cleaned SQL:", sql)`**: Logs the cleaned SQL.
*   **`try...except` block**: Attempts to run the cleaned SQL using `run_query`. If an `Exception` occurs (e.g., an invalid SQL query not caught by `clean_sql`), it catches the error and returns an informative message including the problematic SQL.
*   **`answer = answer_with_data(question, result)`**: This line calls the `answer_with_data` function (which will be defined later) to formulate a natural language answer based on the query result and the original question. This handles the final user-facing output.

In [ ]:
def answer_with_data(question, result):
    prompt = f"""
    Use ONLY the following database result to answer the question.

    Question: {question}

    Database result:
    {result}

    Provide a clear natural-language answer.
    """
    return ai.generate_text(prompt)


In [ ]:
from rdbms_mcp_server import run_query


In [ ]:
# The `run_query` tool from `rdbms_mcp_server` is already imported in cell S0EjKUaM50Ln.
# from rdbms_mcp_server import run_query

def ask_database(question):
    # Step 1: Convert Natural Language (NL) question into an SQL query using the AI model.
    sql = nl_to_sql(question)
    print("Generated SQL:", sql) # Log the raw SQL generated by the AI.

    # Step 2: Clean the generated SQL to remove any markdown formatting or extra characters.
    sql = clean_sql(sql)
    print("Cleaned SQL:", sql) # Log the cleaned SQL.

    # Step 3: Execute the cleaned SQL query against the database.
    try:
        result = run_query(sql)
    except Exception as e:
        # If an error occurs during SQL execution, catch the exception and return an informative message.
        return f"SQL error: {e}\nGenerated SQL was: {sql}"

    # Step 4: Use a separate function (answer_with_data, to be defined) to formulate a natural language answer
    # based on the original question and the database query result.
    answer = answer_with_data(question, result)
    return answer

### 2.7 Testing the Improved `ask_database` Function

We now test our significantly improved `ask_database` function with more complex natural language questions. Observe how it successfully translates the questions into valid SQL, executes them, and provides relevant results, showcasing the power of the NL-to-SQL translation layer.

*   **`print(ask_database("how many products we have"))`**: This question is translated into `SELECT COUNT(DISTINCT product) FROM sales;`, executed, and the count is returned.

In [ ]:
# Test the enhanced `ask_database` function with a natural language query.
# This query asks for the count of distinct products.
print(ask_database("how many products we have"))

### 2.7.1 Querying for Product Price: Case-Sensitive Issue

This test demonstrates querying for the price of a specific product. While `ask_database` successfully translates the question into SQL, it highlights a common database challenge: case sensitivity. The database might not find 'Mouse' if the stored product name is 'mouse', or vice-versa, leading to no results.

*   **`print(ask_database("what is price of Mouse"))`**: The AI correctly generates `SELECT price FROM sales WHERE product = 'Mouse';`.

In [ ]:
# Query for the price of 'Mouse' (with an uppercase 'M').
# This should successfully find the product if it's stored with this casing.
print(ask_database("what is price of Mouse"))

### 2.7.2 Observing Case Sensitivity: No Match Found

Following up on the previous query, this cell explicitly shows the effect of case sensitivity. When asked for 'mouse' (lowercase), the database, by default, performs a case-sensitive match and returns no rows if the product is stored as 'Mouse' (uppercase). This is a crucial point for robust database interaction.

*   **`print(ask_database("what is price of mouse"))`**: The AI generates `SELECT price FROM sales WHERE product = 'mouse';`, which, given our data, yields no results due to the case mismatch. The `answer_with_data` (which we haven't formally defined yet) returns "The database has no matching data." in this scenario.

In [ ]:
# Query for the price of 'mouse' (with a lowercase 'm').
# Given that our data likely stores 'Mouse' with an uppercase 'M',
# this query is expected to return no results due to case sensitivity in SQLite's default behavior for string comparisons.
print(ask_database("what is price of mouse"))

### 2.8 Crafting Natural Language Responses: The `answer_with_data` Function

Raw database results, while accurate, are often not user-friendly. The `answer_with_data` function acts as a post-processing layer, utilizing an AI model to transform structured query results into clear, concise, natural language answers. This completes the human-like interaction cycle.

*   **`answer_with_data(question, result)`**: Takes the original `question` and the `result` from the database query.
*   **`prompt = f"""..."""`**: Constructs a prompt instructing the AI to formulate an answer *only* using the provided database `result`. This prevents hallucination and ensures grounded responses.
*   **`return ai.generate_text(prompt)`**: Calls the AI model to generate the final natural language answer.

In [ ]:
def answer_with_data(question, result):
    # Construct a prompt for the generative AI model.
    # The prompt instructs the AI to use *only* the provided `result` to answer the `question`.
    # This is crucial for grounding the AI's response in factual data and preventing hallucinations.
    prompt = f"""
    Use ONLY the following database result to answer the question.

    Question: {question}

    Database result:
    {result}

    Provide a clear natural-language answer.
    """
    # Call the AI model to generate a natural language answer based on the prompt and data.
    return ai.generate_text(prompt)

### 2.9 Enhancing `ask_database`: Case-Insensitive Fallback

To make our `ask_database` function more robust against case sensitivity issues, we introduce a retry mechanism. If an initial query yields no results, the function attempts a second query using a case-insensitive comparison (e.g., `LOWER(product)`). This significantly improves the user experience by handling common input variations.

*   **`if len(result["rows"]) == 0:`**: Checks if the initial query returned no data.
*   **`sql = nl_to_sql(question + " (use LOWER(product))")`**: If no rows are found, the `nl_to_sql` function is called again with an appended hint to encourage the AI to generate a case-insensitive SQL query (e.g., using `LOWER()`).
*   **`result = run_query(sql)`**: The retry SQL is executed.
*   This ensures that queries like "what is price of mouse" will now correctly find data for 'Mouse'.

### 3.1 Basic Data Visualization: The `create_graph` Function

Numerical data is often best understood visually. The `create_graph` function, utilizing `matplotlib.pyplot`, provides a simple way to generate bar charts from single-column query results. This lays the foundation for integrating visualization directly into our natural language interface.

*   **`import matplotlib.pyplot as plt`**: Imports the plotting library.
*   **`create_graph(result, question)`**: Takes the database `result` (expected to be a list of single-value rows) and the original `question` for the title.
*   **`values = [row[0] for row in result["rows"]]`**: Extracts numerical values from the first column of the query results.
*   **`plt.figure(figsize=(4,3))`**: Sets up a small plot figure.
*   **`plt.bar(["Result"], values)`**: Creates a bar chart. For single-value results, a generic label like "Result" is used.
*   **`plt.title(question)`**: Sets the plot title to the original question.
*   **`plt.show()`**: Displays the generated plot.

In [ ]:
def ask_database(question):
    # Step 1: Convert Natural Language (NL) question into an SQL query using the AI model.
    sql = nl_to_sql(question)
    print("Generated SQL:", sql) # Log the raw SQL generated by the AI.

    # Step 2: Clean the generated SQL to remove any markdown formatting or extra characters.
    sql = clean_sql(sql)
    print("Cleaned SQL:", sql) # Log the cleaned SQL.

    # Step 3: Execute the cleaned SQL query against the database.
    try:
        result = run_query(sql)
    except Exception as e:
        # If an error occurs during SQL execution, catch the exception and return an informative message.
        return f"SQL error: {e}\nGenerated SQL was: {sql}"

    # Step 4: Implement a case-insensitive fallback mechanism.
    # If the initial query returns no rows, attempt a retry with a modified question to encourage case-insensitive SQL generation.
    if len(result["rows"]) == 0:
        print("No rows found. Retrying with case-insensitive SQL...")
        # Append a hint to the question for the AI to generate SQL using `LOWER()` for comparison.
        sql = nl_to_sql(question + " (use LOWER(product))")
        sql = clean_sql(sql)
        print("Retry SQL:", sql) # Log the SQL used for the retry.
        try:
            result = run_query(sql)
        except Exception as e:
            return f"Retry SQL error: {e}\nRetry SQL was: {sql}"

    # Step 5: Use `answer_with_data` to formulate a natural language answer based on the query result.
    answer = answer_with_data(question, result)
    return answer

### 2.9.1 Testing the Case-Insensitive Fallback

This cell re-runs the query for 'mouse' (lowercase) to demonstrate the effectiveness of the new case-insensitive fallback mechanism. Observe how the function now successfully retrieves the price, even with the initial case mismatch, thanks to the intelligent retry logic.

*   **`print(ask_database("what is price of mouse"))`**: The output now shows the retry logic in action and correctly retrieves the price for 'Mouse'.

In [ ]:
# Re-run the query for 'mouse' (lowercase) to test the case-insensitive fallback.
# Expect to see the 'No rows found. Retrying with case-insensitive SQL...' message
# followed by a successful retrieval of the price for 'Mouse'.
print(ask_database("what is price of mouse"))

## Module 3: Data Visualization and Advanced Query Capabilities

In this module, we elevate our database interaction by introducing data visualization. We'll enable our `ask_database` function to generate graphical representations of data, making insights more accessible. We'll start with a basic graphing capability and then extend it to comparative visualizations.

In [ ]:
# Attempt to ask for a graph of the price of 'Mouse'.
# At this stage, `ask_database` will generate the SQL and fetch the data,
# but it lacks the logic to call `create_graph` based on the "graph" keyword in the question.
# Therefore, it will return a text-based answer, not a visual plot.
print(ask_database("create graph for price of Mouse"))

### 3.1.2 Integrating Basic Graph Support into `ask_database`

To enable graph generation via natural language, we modify `ask_database` to detect keywords like "graph" in the user's question. If detected, it will call our `create_graph` function after executing the SQL query. This adds a powerful visualization capability to our natural language interface.

*   **`if "graph" in question.lower():`**: A new conditional check to detect graph requests.
*   **`create_graph(result, question)`**: If a graph is requested, the `create_graph` function is called with the query results and the original question.
*   **`return "Graph created above."`**: A confirmation message is returned to the user, indicating that a graph has been generated.

In [ ]:
import matplotlib.pyplot as plt

def create_graph(result, question):
    # Expecting the 'rows' in the result to be a list of tuples, e.g., `[(25.0,)]` for a single value.
    # Extract the first element from each row to get the numerical values for the bar chart.
    values = [row[0] for row in result["rows"]]

    # Create a new figure for the plot with a specified size (width, height).
    plt.figure(figsize=(4,3))
    # Create a bar chart. For single numerical results, a generic label like 'Result' is used on the x-axis.
    # The actual numerical values are plotted as the height of the bar.
    plt.bar(["Result"], values)
    # Set the title of the plot to the original question.
    plt.title(question)
    # Display the generated plot.
    plt.show()

### 3.1.1 Attempting Basic Graph Creation: Initial Failure

This cell demonstrates an attempt to create a graph for the price of 'Mouse' using the *current* `ask_database` function. It highlights that while we have a `create_graph` function, `ask_database` is not yet intelligent enough to trigger it based on a natural language request. It processes the query but does not call the visualization function, leading to a text-based result instead of a graph.

*   **`print(ask_database("create graph for price of Mouse"))`**: The AI correctly extracts the data but the `ask_database` function is not configured to call `create_graph` yet, hence it produces a text response instead of a plot. The response indicates the data is there, but a graph wasn't created.

In [ ]:
def ask_database(question):
    # Step 1: Convert Natural Language (NL) question into an SQL query using the AI model.
    sql = nl_to_sql(question)
    print("Generated SQL:", sql) # Log the raw SQL generated by the AI.

    # Step 2: Clean the generated SQL.
    sql = clean_sql(sql)
    print("Cleaned SQL:", sql) # Log the cleaned SQL.

    # Step 3: Execute the cleaned SQL query.
    try:
        result = run_query(sql)
    except Exception as e:
        return f"SQL error: {e}\nGenerated SQL was: {sql}"

    # Step 4: Case-insensitive fallback mechanism.
    if len(result["rows"]) == 0:
        print("No rows found. Retrying with case-insensitive SQL...")
        sql = nl_to_sql(question + " (use LOWER(product))")
        sql = clean_sql(sql)
        print("Retry SQL:", sql)
        try:
            result = run_query(sql)
        except Exception as e:
            return f"Retry SQL error: {e}\nRetry SQL was: {sql}"

    # Step 5: Add basic graph support.
    # If the question contains 'graph', call `create_graph` with the result.
    if "graph" in question.lower():
        create_graph(result, question)
        return "Graph created above." # Return a message indicating graph creation.

    # Step 6: Fallback to LLM for natural language answers if no graph was requested.
    answer = answer_with_data(question, result)
    return answer

### 3.1.3 Testing Basic Graph Creation

Now, with `ask_database` updated, we re-test the request for a graph. Observe how it successfully processes the query, generates the SQL, retrieves the data, and then renders a simple bar chart using `create_graph`, providing a visual answer to the natural language question.

In [ ]:
# Test the updated `ask_database` with a request for a graph.
# This should now correctly generate the SQL, fetch the data, and then display a bar chart.
print(ask_database("create graph for price of all products products on x axis and price on y axis"))

### 3.2 Advanced Visualization: The `create_comparison_graph` Function

Beyond simple single-value graphs, comparative visualizations are crucial for understanding relationships between different data points. The `create_comparison_graph` function is designed to handle queries that return two columns (e.g., product names and their prices), plotting them as a bar chart for easy comparison. This function includes robust checks for data types to ensure successful plotting.

*   **`import matplotlib.pyplot as plt`**: Ensures `matplotlib` is imported.
*   **`create_comparison_graph(result, question)`**: Takes the database `result` (expected to be multi-column) and the original `question`.
*   **`columns = result["columns"]`**: Retrieves column names from the result.
*   **`rows = result["rows"]`**: Retrieves data rows.
*   **`if len(columns) < 2:`**: Ensures at least two columns are present for comparison (X and Y axes).
*   **`x_values = [row[0] for row in rows]`**: Extracts values for the X-axis from the first column.
*   **`y_values = [row[1] for row in rows]`**: Extracts values for the Y-axis from the second column.
*   **`try...except` for numeric check**: Attempts to convert Y-axis values to floats, ensuring they are numeric. Handles non-numeric data gracefully.
*   **`plt.bar(x_values, y_values, color='skyblue')`**: Creates the bar chart.
*   **`plt.xlabel(columns[0])`, `plt.ylabel(columns[1])`, `plt.title(question)`**: Labels the axes and sets the title.
*   **`plt.xticks(rotation=45)`**: Rotates X-axis labels to prevent overlap.
*   **`plt.tight_layout()`**: Adjusts plot parameters for a tight layout.
*   **`plt.show()`**: Displays the plot.

In [ ]:
import matplotlib.pyplot as plt

def create_comparison_graph(result, question):
    # Extract column names and rows from the database query result.
    columns = result["columns"]
    rows = result["rows"]

    # A comparison graph typically requires at least two columns: one for categories (x-axis) and one for values (y-axis).
    if len(columns) < 2:
        print("Not enough columns to create a comparison graph. At least two columns are required.")
        return

    # Extract values for the x-axis from the first column of the result.
    x_values = [row[0] for row in rows]

    # Extract values for the y-axis from the second column of the result.
    y_values = [row[1] for row in rows]

    # Attempt to convert y-axis values to numeric (float). This is crucial for plotting.
    # Handle cases where values might not be convertible to numbers.
    try:
        y_values = [float(v) for v in y_values]
    except ValueError:
        print("Y-axis values are not numeric. Cannot plot graph.")
        return

    # Create a new figure for the plot with a specified size.
    plt.figure(figsize=(6,4))
    # Generate a bar chart using the extracted x and y values.
    plt.bar(x_values, y_values, color='skyblue')
    # Label the x-axis with the name of the first column.
    plt.xlabel(columns[0])
    # Label the y-axis with the name of the second column.
    plt.ylabel(columns[1])
    # Set the title of the plot to the original question.
    plt.title(question)
    # Rotate x-axis labels by 45 degrees to prevent overlapping, especially with long labels.
    plt.xticks(rotation=45)
    # Adjust plot parameters for a tight layout, preventing labels or titles from being cut off.
    plt.tight_layout()
    # Display the generated plot.
    plt.show()

### 3.2.1 Integrating Comparison Graph Support into `ask_database`

This final iteration of `ask_database` combines all our capabilities: NL-to-SQL translation, case-insensitive fallback, basic graphing, and now, advanced comparison graphing. It prioritizes comparison graph requests, then general graph requests, and finally falls back to AI-generated natural language answers. This creates a highly versatile and intelligent database interaction agent.

*   **`if "compare graph" in question.lower() or "comparison graph" in question.lower():`**: A new conditional check for comparison graph requests. It looks for either phrase.
*   **`create_comparison_graph(result, question)`**: If a comparison graph is requested, this specific function is called.
*   **`return "Comparison graph created above."`**: A confirmation message for comparison graphs.
*   The order of checks (`compare graph` first, then `graph`, then `answer_with_data`) is important to ensure the most specific visualization request is handled correctly.

In [ ]:
def ask_database(question):
    # Step 1: Convert Natural Language (NL) question into an SQL query using the AI model.
    sql = nl_to_sql(question)
    print("Generated SQL:", sql)

    # Step 2: Clean the generated SQL.
    sql = clean_sql(sql)
    print("Cleaned SQL:", sql)

    # Step 3: Execute the cleaned SQL query.
    try:
        result = run_query(sql)
    except Exception as e:
        return f"SQL error: {e}\nGenerated SQL was: {sql}"

    # Step 4: Case-insensitive fallback mechanism.
    if len(result["rows"]) == 0:
        print("No rows found. Retrying with case-insensitive SQL...")
        sql = nl_to_sql(question + " (use LOWER(product))")
        sql = clean_sql(sql)
        print("Retry SQL:", sql)
        try:
            result = run_query(sql)
        except Exception as e:
            return f"Retry SQL error: {e}\nRetry SQL was: {sql}"

    # Step 5: Implement advanced comparison graph support.
    # Check if the question explicitly asks for a 'compare graph' or 'comparison graph'.
    if "compare graph" in question.lower() or "comparison graph" in question.lower():
        create_comparison_graph(result, question)
        return "Comparison graph created above."

    # Step 6: Implement basic graph support (if not a comparison graph).
    # If the question contains 'graph', call `create_graph`.
    if "graph" in question.lower():
        create_graph(result, question)
        return "Graph created above."

    # Step 7: Fallback to LLM for natural language answers if no graph was requested.
    answer = answer_with_data(question, result)
    return answer

### 3.2.2 Testing Comparison Graph Creation

Finally, we test the ultimate capability of our `ask_database` function: creating a comparison graph from a natural language query. This demonstrates the seamless integration of AI understanding, SQL generation, data retrieval, and advanced visualization, providing a comprehensive and intuitive way to explore database insights.

In [ ]:
# Test the `ask_database` function with a request for a comparison graph.
# This should trigger the NL-to-SQL translation, data retrieval, and then the `create_comparison_graph` function.
print(ask_database("create compare graph for price of all products products on x axis and price on y axis"))